# PDAC A-1: full MORSE workflow

Follow the complete main-model workflow from raw counts to additional-location
expression. PDAC uses 428 measured spots from GSM3036911 and the registered working H&E image. The source registration columns are explicitly converted to image (x, y).

Install the dependencies using the project README before opening this notebook.
This notebook is distributed without stored outputs or fitted weights. It calls
the same source functions as the command-line entry, one step at a time.

**Plan:** inputs → normalized expression → H&E features → fit MORSE → new
locations → inference → save → optional display. All supplied measured tissue
locations are used for fitting; this is not a held-out accuracy benchmark.


## 1. Find the project and read the configuration

Paths are relative to the extracted MORSE folder. Edit the configuration file
or the overrides in this cell; no private server path is required.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np

here = Path.cwd().resolve()
ROOT = next((p for p in (here, *here.parents)
             if (p / "src/morse/model.py").is_file() and (p / "configs/pdac.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the extracted MORSE project.")
sys.path.insert(0, str(ROOT / "src"))

from morse.config import read_config, MORSEConfig
from morse.data import load_dataset
from morse.preprocessing import prepare_expression
from morse.image_features import rgb_features
from morse.training import fit_morse, predict_expression
from morse.locations import candidate_locations
from morse.output import make_prediction_adata, save_run
from morse.visualization import plot_input, plot_expression

config = read_config(ROOT / "configs/pdac.json")
# Optional overrides; leave these unchanged for the supplied configuration.
# config["model"]["device"] = "cuda:0"
# config["model"]["epochs"] = 5000
OUTPUT_DIR = ROOT / config["output_dir"]
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError("Choose a new OUTPUT_DIR before starting another run.")
print("Project:", ROOT)
print("Output:", OUTPUT_DIR)
print(json.dumps(config, indent=2))


## 2. Read raw counts, registered coordinates and H&E

The loader retains sample IDs and aligns all rows by spot ID or barcode.
Counts are still unnormalized at this step. The source image and data
provenance are documented in `docs/DATA.md`.


In [ ]:
data = load_dataset(config, ROOT)
print("Sample:", data.sample_id)
print("Raw count shape (spots, genes):", data.counts.shape)
print("Loaded image shape:", data.image.shape)
print("Registered-coordinate to image scale:", data.image_scale)
display(data.counts.obs.head())


## 3. Inspect registration

This optional display overlays measured spot centres on the image. Feature
extraction below uses the same image scale. This is an input check, not an
experimental result bundled with the repository.


In [ ]:
registration_figure = plot_input(data)
display(registration_figure)


## 4. Normalize measured counts and select genes

Filter nonempty measured spots and genes detected in at least five spots;
divide by each spot's original count total, multiply by 10,000, apply log1p,
then select up to 3,000 genes by variance. The returned matrix is in normalized
log1p units. This function accepts counts, not previously normalized H5AD values.


In [ ]:
prepared = prepare_expression(data.counts, config["preprocessing"])
print(json.dumps(prepared.metadata, indent=2))
display(prepared.observed.var.head())


## 5. Extract image descriptors at measured locations

Each image patch contributes RGB means and SD (six values). Patches are
15 × 15 pixels in the loaded working/hires image. Coordinate and feature
z-score parameters will be fitted on the measured locations inside training.


In [ ]:
measured_features = rgb_features(
    data.image, prepared.coordinates * data.image_scale,
    patch_size=config["image"]["patch_size"],
)
print("Expression:", prepared.expression.shape)
print("Coordinates:", prepared.coordinates.shape)
print("Image features:", measured_features.shape)


## 6. Fit the full MORSE model

This cell performs training when you execute it. E, D, f and phi are updated
jointly using the five objectives. The graph and molecular affinity targets
contain only the expression-bearing measured locations supplied here.
No expression at any new candidate location is available or required.


In [ ]:
model_config = MORSEConfig(**config["model"])
model, state, history = fit_morse(
    prepared.expression, prepared.coordinates, measured_features, model_config,
)


## 7. Propose new supported locations

PDAC uses a d/2 grid; Visium uses three half-step native lattice offsets.
Candidates are deduplicated and filtered by convex hull and nearest measured
distance. This coordinate filter is not a tissue-hole segmentation mask.
The printed metadata reports the actual achieved density after filtering.


In [ ]:
candidates, generation_metadata = candidate_locations(
    prepared.coordinates, prepared.observed.obs,
    mode=config["generation"]["mode"],
    density_factor=config["generation"]["density_factor"],
    seed=config["model"]["seed"],
)
new_features = rgb_features(
    data.image, candidates * data.image_scale,
    patch_size=config["image"]["patch_size"],
)
print(json.dumps(generation_metadata, indent=2))


## 8. Generate expression on measured and new locations

Inference builds a graph on O ∪ U using coordinates and image features.
The first rows correspond to measured coordinates and the remaining rows to
new coordinates. All returned expression rows are model predictions;
measured expression remains in `prepared.observed`.


In [ ]:
prediction, intrinsic = predict_expression(model, state, candidates, new_features)
result = make_prediction_adata(
    prepared, prediction, intrinsic, candidates,
    np.vstack((measured_features, new_features)),
)
print(result)
print(result.obs["location_type"].value_counts())


## 9. Save your run

Saves normalized measured expression, generated predictions, coordinates,
selected genes, loss history, fitted model and effective configuration.
Nothing is overwritten in a nonempty output folder.


In [ ]:
save_run(
    OUTPUT_DIR, config, data, prepared, model, state, history,
    result, generation_metadata,
)


## 10. Optional expression display

Choose a selected gene. Both panels share a colour scale. The left panel
contains measured expression; the right contains dense model predictions.
This display is not a test of accuracy at unmeasured biological locations.
To export vector graphics, uncomment the final `savefig` line.


In [ ]:
preferred_gene = "KRT17"
gene = preferred_gene if preferred_gene in result.var_names else str(result.var_names[0])
expression_figure = plot_expression(prepared, result, gene)
display(expression_figure)
# expression_figure.savefig(OUTPUT_DIR / f"{gene}_expression.svg", bbox_inches="tight")


## Next: reuse the fitted model

Inspect `selected_genes.csv` and choose another gene for the optional display.
To predict a new coordinate set without retraining, see
`scripts/generate_from_checkpoint.py` and README section 6. Supply matching
registered x/y coordinates; no query-expression column is needed.
